# Lakebase setup

Idempotent job that ensures the agent-dock app's service principal has full access to the registry schema in Lakebase. Run after `databricks bundle deploy` and before `databricks bundle run registry_app`.

Behavior:
- Schema missing: no-op. The app's `bootstrap_schema` will create it on first start as the SP.
- Schema owned by another principal (e.g. a user from a notebook-based setup): grants USAGE + CRUD on schema + tables to the SP. Idempotent.
- Schema already owned by the SP: no-op.

In [ ]:
%pip install -q 'psycopg[binary]'
dbutils.library.restartPython()

In [ ]:
import uuid
from urllib.parse import quote_plus

import psycopg
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("app_name", "mcp-agent-dock")
dbutils.widgets.text("instance_name", "agent-registry")
dbutils.widgets.text("schema", "agent_registry")

app_name = dbutils.widgets.get("app_name")
instance_name = dbutils.widgets.get("instance_name")
schema = dbutils.widgets.get("schema")

ws = WorkspaceClient()
sp_id = ws.apps.get(name=app_name).service_principal_client_id
inst = ws.database.get_database_instance(name=instance_name)
user = ws.current_user.me().user_name
cred = ws.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[instance_name],
)
token = cred.token
dsn = (
    f"postgresql://{quote_plus(user)}:{quote_plus(token)}@"
    f"{inst.read_write_dns}:5432/databricks_postgres?sslmode=require"
)
print(f"app={app_name} sp={sp_id} instance={instance_name} schema={schema}")

In [ ]:
with psycopg.connect(dsn, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute(
        "SELECT pg_get_userbyid(nspowner) FROM pg_namespace WHERE nspname = %s",
        (schema,),
    )
    row = cur.fetchone()
    if row is None:
        print(f"schema {schema} not present; app will create it on startup as {sp_id}")
    elif row[0] == sp_id:
        print(f"schema {schema} already owned by {sp_id}; nothing to do")
    else:
        for sql in [
            f'GRANT USAGE ON SCHEMA "{schema}" TO "{sp_id}"',
            f'GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA "{schema}" TO "{sp_id}"',
            f'GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA "{schema}" TO "{sp_id}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA "{schema}" '
            f'GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO "{sp_id}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA "{schema}" '
            f'GRANT USAGE, SELECT ON SEQUENCES TO "{sp_id}"',
        ]:
            cur.execute(sql)
        print(f"schema {schema} owned by {row[0]}; granted USAGE+CRUD to {sp_id}")